# Customer revenue pipeline — baseline model

Train one global linear model from the preprocessing notebook's train/test split.

In [ ]:
# @node: Train baseline  [transform]  in=train_df<-preprocessing:Train test split.train_df,test_df<-preprocessing:Train test split.test_df,feature_cols<-preprocessing:Train test split.feature_cols  out=baseline_model,baseline_scores,baseline_predictions
import numpy as np
import pandas as pd

def design_matrix(df, columns):
    x = df[columns].to_numpy(dtype=float)
    return np.column_stack([np.ones(len(df)), x])

x_train = design_matrix(train_df, feature_cols)
y_train = train_df["revenue"].to_numpy(dtype=float)
coef = np.linalg.lstsq(x_train, y_train, rcond=None)[0]

x_test = design_matrix(test_df, feature_cols)
pred = x_test @ coef
actual = test_df["revenue"].to_numpy(dtype=float)
err = pred - actual

baseline_predictions = test_df[["date", "channel", "region", "revenue"]].copy()
baseline_predictions["prediction"] = pred.round(2)
baseline_predictions["model"] = "global linear baseline"

baseline_scores = pd.DataFrame(
    [
        {
            "model": "global linear baseline",
            "rmse": float(np.sqrt(np.mean(err**2))),
            "mae": float(np.mean(np.abs(err))),
            "bias": float(np.mean(err)),
        }
    ]
)
baseline_model = {
    "kind": "global_linear",
    "intercept": float(coef[0]),
    "coefficients": dict(zip(feature_cols, coef[1:].round(4), strict=True)),
}
display(baseline_scores.round(2))